calendar and availability - adding tables

In [0]:
from pyspark.sql import functions as F

cal_b    = spark.table("retail_inv.bronze.calendar")
prices_b = spark.table("retail_inv.bronze.sell_prices")
sales_b  = spark.table("retail_inv.bronze.sales_wide")

cal = (cal_b
    .withColumn("date", F.to_date("date"))
    .select("date","d","wm_yr_wk","weekday","wday","month","year",
            "event_name_1","event_type_1","event_name_2","event_type_2",
            "snap_CA","snap_TX","snap_WI"))
cal.write.format("delta").mode("overwrite").saveAsTable("retail_inv.silver.calendar")

wk_start = cal.groupBy("wm_yr_wk").agg(F.min("date").alias("week_start_date"))

avail = (prices_b.groupBy("store_id","item_id")
    .agg(F.min("wm_yr_wk").alias("first_priced_wk"))
    .join(wk_start, F.col("first_priced_wk") == wk_start.wm_yr_wk)
    .select("store_id","item_id","first_priced_wk",
            F.col("week_start_date").alias("first_priced_date")))
avail.write.format("delta").mode("overwrite").saveAsTable("retail_inv.silver.item_store_availability")

In [0]:
for table_name in [
    "retail_inv.silver.calendar",
    "retail_inv.silver.item_store_availability"
]:
    df = spark.table(table_name)
    print(f"{table_name}: {df.count():,} rows")
    

retail_inv.silver.calendar: 1,969 rows
retail_inv.silver.item_store_availability: 30,490 rows


Build Sales Daily

In [0]:
from pyspark.sql import functions as F

# Identify the six descriptive columns and 1,913 daily sales columns.
ids = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

d_cols = sorted(
    [
        c for c in sales_b.columns
        if c.startswith("d_") and c[2:].isdigit()
    ],
    key=lambda c: int(c[2:])
)

print("Daily columns identified:", len(d_cols))

# Convert the wide sales table into one row per item-store-day.
sales_daily = (
    sales_b
    .unpivot(ids, d_cols, "d", "units_demand")

    # Attach the calendar date and retail week.
    .join(
        cal.select("d", "date", "wm_yr_wk"),
        on="d",
        how="inner"
    )

    # Attach the weekly recorded selling price.
    .join(
        prices_b.select(
            "store_id", "item_id", "wm_yr_wk", "sell_price"
        ),
        on=["store_id", "item_id", "wm_yr_wk"],
        how="left"
    )

    # Treat the presence of a recorded price as an availability proxy.
    .withColumn(
        "is_available",
        F.col("sell_price").isNotNull()
    )

    # Estimated revenue for each item-store-day.
    .withColumn(
        "revenue",
        F.col("units_demand") * F.col("sell_price")
    )

    .select(
        "item_id",
        "store_id",
        "dept_id",
        "cat_id",
        "state_id",
        "date",
        "d",
        "wm_yr_wk",
        "units_demand",
        "sell_price",
        "is_available",
        "revenue"
    )
)

# Save the result as a Silver Delta table.
(
    sales_daily.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("state_id")
    .saveAsTable("retail_inv.silver.sales_daily")
)

print("Silver sales_daily table created successfully!")

Daily columns identified: 1913
Silver sales_daily table created successfully!


In [0]:
display(sales_daily)

item_id,store_id,dept_id,cat_id,state_id,date,d,wm_yr_wk,units_demand,sell_price,is_available,revenue
FOODS_1_201,TX_2,FOODS_1,FOODS,TX,2011-01-31,d_3,11101,6,2.0,true,12.0
FOODS_1_209,TX_2,FOODS_1,FOODS,TX,2011-01-31,d_3,11101,2,2.38,true,4.76
FOODS_2_029,TX_2,FOODS_2,FOODS,TX,2011-01-31,d_3,11101,5,10.48,true,52.400000000000006
FOODS_2_031,TX_2,FOODS_2,FOODS,TX,2011-01-31,d_3,11101,4,1.88,true,7.52
FOODS_2_144,TX_2,FOODS_2,FOODS,TX,2011-01-31,d_3,11101,2,2.48,true,4.96
FOODS_2_232,TX_2,FOODS_2,FOODS,TX,2011-01-31,d_3,11101,0,1.96,true,0.0
FOODS_2_237,TX_2,FOODS_2,FOODS,TX,2011-01-31,d_3,11101,1,2.48,true,2.48
FOODS_2_340,TX_2,FOODS_2,FOODS,TX,2011-01-31,d_3,11101,4,2.96,true,11.84
FOODS_3_001,TX_2,FOODS_3,FOODS,TX,2011-01-31,d_3,11101,0,2.28,true,0.0
FOODS_3_050,TX_2,FOODS_3,FOODS,TX,2011-01-31,d_3,11101,2,1.48,true,2.96


Validation

In [0]:
from pyspark.sql import functions as F

t = spark.table("retail_inv.silver.sales_daily")

# Validate the main Silver-layer metrics in one scan.
metrics = t.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(~F.col("is_available"), 1).otherwise(0)
    ).alias("unavailable_rows"),

    F.sum(
        F.when(F.col("is_available"), 1).otherwise(0)
    ).alias("available_rows"),

    F.sum(
        F.when(
            F.col("is_available") & (F.col("units_demand") == 0),
            1
        ).otherwise(0)
    ).alias("zero_sales_available"),

    F.sum(
        F.when(
            (F.col("units_demand") > 0) & (~F.col("is_available")),
            1
        ).otherwise(0)
    ).alias("positive_sales_without_price"),

    F.sum(
        F.when(F.col("units_demand") < 0, 1).otherwise(0)
    ).alias("negative_sales_rows")
).first()

print(f"Total rows: {metrics['total_rows']:,}")
print(f"Unavailable rows: {metrics['unavailable_rows']:,}")
print(f"Available rows: {metrics['available_rows']:,}")
print(f"Positive sales without a price: {metrics['positive_sales_without_price']:,}")
print(f"Negative sales rows: {metrics['negative_sales_rows']:,}")

available_rows = metrics["available_rows"] or 0
zero_sales = metrics["zero_sales_available"] or 0

if available_rows:
    print(
        f"Zero-sales share on available rows: "
        f"{zero_sales / available_rows * 100:.2f}%"
    )

Total rows: 58,327,370
Unavailable rows: 12,299,413
Available rows: 46,027,957
Positive sales without a price: 0
Negative sales rows: 0
Zero-sales share on available rows: 59.70%
